# 16. Finding Variants: SNPs, Indels and the VCF Format

The programs of this chapter, in the order the book prints
them. Each cell holds the file of the same name in this folder,
so editing a cell does not change the script beside it.

Run the cells from this folder, where the data files the chapter
uses live. Some cells call command line tools or download data,
and a few take minutes.


## `naive_caller.py`

The simplest possible variant caller: at each position, report the most common base if enough reads cover it and enough of them agree.


In [ ]:
# The simplest possible variant caller: at each position, report the most
# common base if enough reads cover it and enough of them agree.
import sys

import numpy as np
import pysam
from Bio import SeqIO

MIN_DEPTH = int(sys.argv[1]) if len(sys.argv) > 1 else 10
MIN_SHARE = float(sys.argv[2]) if len(sys.argv) > 2 else 0.8

reference = np.array(
    list(SeqIO.read("../data/rel606.fasta", "fasta").seq)
)
bam = pysam.AlignmentFile("../ch15/reads.bam")
counts = np.array(
    bam.count_coverage(
        bam.references[0], 0, len(reference), quality_threshold=0
    )
)

depth = counts.sum(axis=0)
top_base = np.array(list("ACGT"))[counts.argmax(axis=0)]
share = counts.max(axis=0) / np.maximum(depth, 1)
called = (depth >= MIN_DEPTH) & (share >= MIN_SHARE)
called &= top_base != reference

positions = np.where(called)[0]
print(
    f"depth >= {MIN_DEPTH}, share >= {MIN_SHARE}: {len(positions)} calls"
)
rows = [
    f"{p + 1}\t{reference[p]}\t{top_base[p]}\t{depth[p]}\n"
    for p in positions
]
with open("naive.tsv", "w") as out:
    out.writelines(rows)

## `compare.py`

Compare the naive caller's single letter calls with bcftools'.


In [ ]:
# Compare the naive caller's single letter calls with bcftools'.
import pysam

with open("naive.tsv") as handle:
    naive = {int(line.split("\t")[0]) for line in handle}
bcftools = {
    record.pos
    for record in pysam.VariantFile("haploid.vcf.gz")
    if record.qual >= 30
    and record.info["DP"] >= 10
    and len(record.ref) == 1
    and len(record.alts[0]) == 1
}
print(
    f"naive {len(naive)}, bcftools {len(bcftools)}, "
    f"both {len(naive & bcftools)}"
)
print(f"only bcftools: {sorted(bcftools - naive)}")
extra = sorted(naive - bcftools)
if extra:
    print(f"only naive: {len(extra)}, the first at {extra[:5]}")
else:
    print("only naive: none")

## `effects.py`

What each confident variant does: which gene it falls in and, for single letter changes in a coding sequence, the amino acid change.


In [ ]:
# What each confident variant does: which gene it falls in and, for
# single letter changes in a coding sequence, the amino acid change.
import pandas as pd
import pysam
from Bio import SeqIO
from Bio.Seq import Seq

reference = SeqIO.read("../data/rel606.fasta", "fasta").seq
columns = [
    "seqid",
    "source",
    "type",
    "start",
    "end",
    "score",
    "strand",
    "phase",
    "attributes",
]
gff = pd.read_csv(
    "../ch15/rel606.gff3", sep="\t", comment="#", names=columns
)
cds = gff[gff["type"] == "CDS"].drop_duplicates(subset=["start", "end"])
cds = cds.assign(
    gene=cds["attributes"].str.extract(r"gene=([^;]+)", expand=False),
    product=cds["attributes"].str.extract(
        r"product=([^;]+)", expand=False
    ),
)


def effect(row, position: int, alt: str) -> str:
    """The amino acid change a single letter change makes in one CDS."""
    start, end = row.start - 1, row.end  # zero based, end excluded
    gene = reference[start:end]
    offset = position - 1 - start
    if row.strand == "-":
        gene = gene.reverse_complement()
        offset = end - position
        alt = str(Seq(alt).reverse_complement())
    mutated = gene[:offset] + alt + gene[offset + 1 :]
    codon = offset // 3
    before = gene[3 * codon : 3 * codon + 3].translate(table=11)
    after = mutated[3 * codon : 3 * codon + 3].translate(table=11)
    kind = (
        "synonymous"
        if before == after
        else ("nonsense" if after == "*" else "missense")
    )
    return f"{before}{codon + 1}{after} {kind}"


for record in pysam.VariantFile("haploid.vcf.gz"):
    if record.qual < 30 or record.info["DP"] < 10:
        continue
    alt = record.alts[0]
    hits = cds[(cds["start"] <= record.pos) & (cds["end"] >= record.pos)]
    if hits.empty:
        where = "between genes"
    else:
        row = hits.iloc[0]
        name = (
            row.gene if isinstance(row.gene, str) else row["product"][:24]
        )
        if "IS1" in row["product"]:
            name = "IS1 element"
        if "frameshift" in row["attributes"]:
            where = f"{name}: frameshifted gene, not translated"
        elif len(record.ref) == 1 and len(alt) == 1:
            where = f"{name}: {effect(row, record.pos, alt)}"
        else:
            where = f"{name}: insertion or deletion"
    change = f"{record.ref[:4]}>{alt[:4]}"
    print(f"{record.pos:9,}  {change:10} {where}")

## `clinvar_hbb.py`

What ClinVar records about variants in HBB, fetched by region from NCBI's indexed ClinVar file without downloading all of it.


In [ ]:
# What ClinVar records about variants in HBB, fetched by region from
# NCBI's indexed ClinVar file without downloading all of it.
from collections import Counter
from pathlib import Path

import pysam

CLINVAR = (
    "https://ftp.ncbi.nlm.nih.gov/pub/clinvar/vcf_GRCh38/clinvar.vcf.gz"
)
HBB = ("11", 5_225_463, 5_227_071)  # GRCh38, zero based start
SAVED = Path("hbb_clinvar.vcf")

if not SAVED.exists():
    with (
        pysam.VariantFile(CLINVAR) as remote,
        pysam.VariantFile(SAVED, "w", header=remote.header) as out,
    ):
        for record in remote.fetch(*HBB):
            out.write(record)


def readable(values: tuple) -> str:
    """ClinVar's ("criteria_provided", "_multiple_submitters") as text."""
    return ",".join(values).replace("_", " ").strip()


records = list(pysam.VariantFile(SAVED))
print(f"{len(records)} ClinVar records in HBB")

significance = Counter(
    r.info.get("CLNSIG", ("none",))[0] for r in records
)
for label, count in significance.most_common(6):
    print(f"  {readable((label,)):44} {count:5}")

sickle = next(r for r in records if r.info.get("RS") == ("334",))
change = f"{sickle.ref}>{sickle.alts[0]}"
print(f"\nrs334 at chromosome {sickle.chrom}, {sickle.pos:,}, {change}")
print(f"  significance  {readable(sickle.info['CLNSIG'])}")
print(f"  review status {readable(sickle.info['CLNREVSTAT'])}")
print(f"  description   {sickle.info['CLNHGVS'][0]}")
print(f"  consequence   {sickle.info['MC'][0]}")